# XGBoost Training Script for Google Colab

This notebook runs the enhanced XGBoost training code directly on Google Colab servers using locally available training scripts. The script trains models on 50,000+ historical candles for multiple timeframes (15m, 1h, 4h).

**Key Features:**
- GPU acceleration support
- Configurable candle count (20,000-100,000+)
- Multi-timeframe training (15m, 1h, 4h)
- Comprehensive logging and metrics
- Automatic model validation and persistence

**Expected Runtime:** ~30-45 minutes depending on candle count and GPU availability


## 1. Upload Local Scripts

First, upload the local training scripts from your machine. Use the file picker to select train_xgboost_colab.py and delta_historical_fetcher.py.

In [6]:
import sys
from pathlib import Path

# Direct absolute path to project root
# Notebook is at: D:\ATTRAL\Projects\Trading Agent 2\notebooks\train_xgboost_colab.ipynb
PROJECT_ROOT = Path(r"D:\ATTRAL\Projects\Trading Agent 2")

# Add project root to sys.path if not already present
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Define all paths using the project root
XGBOOST_DIR = PROJECT_ROOT / "agent" / "model_storage" / "xgboost"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
DATA_DIR = PROJECT_ROOT / "data"
DATA_DELTA_DIR = DATA_DIR / "backups" / "delta"
TRAINING_SUMMARY_FILE = XGBOOST_DIR / "training_summary.csv"
FEATURE_SCHEMA_FILE = XGBOOST_DIR / "feature_schema.json"
DELTA_FETCHER_SCRIPT = SCRIPTS_DIR / "delta_historical_fetcher.py"

# Create directories
XGBOOST_DIR.mkdir(parents=True, exist_ok=True)
DATA_DELTA_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Project Paths Initialized (Local)")
print(f"  Project Root: {PROJECT_ROOT}")
print(f"  Models Dir: {XGBOOST_DIR}")
print(f"  Scripts Dir: {SCRIPTS_DIR}")
print(f"  Data Dir: {DATA_DELTA_DIR}")

✓ Project Paths Initialized (Local)
  Project Root: D:\ATTRAL\Projects\Trading Agent 2
  Models Dir: D:\ATTRAL\Projects\Trading Agent 2/agent/model_storage/xgboost
  Scripts Dir: D:\ATTRAL\Projects\Trading Agent 2/scripts
  Data Dir: D:\ATTRAL\Projects\Trading Agent 2/data/backups/delta


## 2. Install Dependencies

Install all required Python packages for the training script.

In [7]:
# Verify scripts are accessible
print("\n✓ Attempting to import from scripts...")

try:
    from train_xgboost_colab import train_model
    from delta_historical_fetcher import fetch_historical_data
    print("✓ Successfully imported functions from scripts!")
    print("  - train_model (from train_xgboost_colab)")
    print("  - fetch_historical_data (from delta_historical_fetcher)")
except ImportError as e:
    print(f"⚠ Import note: {e}")
    print("\nYou can also run scripts directly with Python:")
    print("  !python ../scripts/train_xgboost_colab.py")
    print("  !python ../scripts/delta_historical_fetcher.py")


✓ Attempting to import from scripts...
⚠ Import note: No module named 'train_xgboost_colab'

You can also run scripts directly with Python:
  !python ../scripts/train_xgboost_colab.py
  !python ../scripts/delta_historical_fetcher.py


In [8]:
import subprocess
import sys

# Install required packages
packages = ['ta', 'pandas', 'requests', 'xgboost', 'scikit-learn', 'numpy', 'matplotlib', 'torch']
print("Installing required packages...")
for package in packages:
    print(f"  Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
print("✓ All packages installed successfully!")

Installing required packages...
  Installing ta...
  Installing pandas...
  Installing requests...
  Installing xgboost...
  Installing scikit-learn...
  Installing numpy...
  Installing matplotlib...
  Installing torch...
✓ All packages installed successfully!


In [9]:
# 2. Install Dependencies
# All required Python packages have been installed in the Setup cell above.
# If you encounter import errors, run the Setup cell again.
print("✓ Dependencies ready for training")

✓ Dependencies ready for training


## 3. Set Environment Variables

Configure the training script parameters. Adjust as needed for your preferences.

In [10]:
import os
from pathlib import Path

# Use paths initialized in the setup cell above
save_dir = XGBOOST_DIR
save_dir.mkdir(parents=True, exist_ok=True)

# Configuration parameters for training
os.environ['TRAIN_CANDLES'] = '50000'      # Number of candles per timeframe (20000-100000+)
os.environ['TRAIN_SYMBOL'] = 'BTCUSD'      # Trading symbol
os.environ['TRAIN_BATCH_SIZE'] = '500'     # API batch size for fetching
os.environ['TRAIN_NO_GPU'] = ''            # Leave empty to enable GPU, set to '1' to disable
os.environ['TRAIN_SAVE_DIR'] = str(save_dir)

print('Environment Variables Set:')
print(f'  TRAIN_CANDLES:     {os.environ["TRAIN_CANDLES"]}')
print(f'  TRAIN_SYMBOL:      {os.environ["TRAIN_SYMBOL"]}')
print(f'  TRAIN_BATCH_SIZE:  {os.environ["TRAIN_BATCH_SIZE"]}')
print(f'  TRAIN_SAVE_DIR:    {os.environ["TRAIN_SAVE_DIR"]}')
print()

# Check if GPU is available
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f'  GPU:               YES - {torch.cuda.get_device_name(0)}')
    else:
        print(f'  GPU:               NO - will use CPU')
except:
    print(f'  GPU:               PyTorch not loaded')

Environment Variables Set:
  TRAIN_CANDLES:     50000
  TRAIN_SYMBOL:      BTCUSD
  TRAIN_BATCH_SIZE:  500
  TRAIN_SAVE_DIR:    D:\ATTRAL\Projects\Trading Agent 2/agent/model_storage/xgboost

  GPU:               NO - will use CPU


## 4. Run Embedded Training

Execute the XGBoost training pipeline directly in this notebook (no external scripts required). This cell contains all the training logic:

1. Fetch historical data from Delta Exchange API for each timeframe
2. Compute all 50 technical indicators 
3. Generate label signals (SELL/HOLD/BUY)
4. Train XGBoost classifiers with temporal validation split
5. Save trained models with feature validation to Google Drive
6. Generate training summary report

**Note:** This may take 30-45 minutes depending on GPU availability, candle count, and API response times.

In [12]:
# 4. Run Embedded Training
# Execute the XGBoost training pipeline (embedded in this notebook - no external scripts required)
# This cell contains all the training logic:
# 1. Fetch historical data from Delta Exchange API for each timeframe
# 2. Compute all 50 technical indicators 
# 3. Generate label signals (SELL/HOLD/BUY)
# 4. Train XGBoost classifiers with temporal validation split
# 5. Save trained models with feature validation locally
# 6. Generate training summary report
# Note: This may take 30-45 minutes depending on GPU availability, candle count, and API response times.

print("Ready to execute training pipeline. Run the next cell to begin.")

Ready to execute training pipeline. Run the next cell to begin.


In [13]:
#!/usr/bin/env python3
"""
Embedded XGBoost Training — all logic inline (no external script required)
Trains models on 50,000+ historical candles for multiple timeframes.
"""
import logging
import os
import pickle
import math
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Tuple
from collections import Counter

import numpy as np
import pandas as pd
import requests
import ta
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("train_xgboost_inline")

# Canonical 50-feature list (MUST match agent expectations)
FEATURE_LIST: List[str] = [
    "sma_10", "sma_20", "sma_50", "sma_100", "sma_200",
    "ema_12", "ema_26", "ema_50",
    "close_sma_20_ratio", "close_sma_50_ratio", "close_sma_200_ratio",
    "high_low_spread", "close_open_ratio", "body_size", "upper_shadow", "lower_shadow",
    "rsi_14", "rsi_7", "stochastic_k_14", "stochastic_d_14",
    "williams_r_14", "cci_20", "roc_10", "roc_20",
    "momentum_10", "momentum_20",
    "macd", "macd_signal", "macd_histogram",
    "adx_14", "aroon_up", "aroon_down", "aroon_oscillator",
    "trend_strength",
    "bb_upper", "bb_lower", "bb_width", "bb_position",
    "atr_14", "atr_20",
    "volatility_10", "volatility_20",
    "volume_sma_20", "volume_ratio", "obv",
    "volume_price_trend", "accumulation_distribution", "chaikin_oscillator",
    "returns_1h", "returns_24h",
]
EXPECTED_FEATURE_COUNT = len(FEATURE_LIST)

# Configuration
BASE_URL = "https://api.india.delta.exchange"
RESOLUTION_MAP = {
    "1m": "1m", "5m": "5m", "15m": "15m", "30m": "30m",
    "1h": "1h", "2h": "2h", "4h": "4h", "1d": "1d",
}
INTERVAL_SECONDS = {
    "1m": 60, "5m": 300, "15m": 900, "30m": 1800,
    "1h": 3600, "2h": 7200, "4h": 14400, "1d": 86400,
}
RETURNS_1H_BARS = {"15m": 4, "1h": 1, "4h": 1}
RETURNS_24H_BARS = {"15m": 96, "1h": 24, "4h": 6}
BUY_THRESHOLD, SELL_THRESHOLD = 0.5, -0.5

# Check GPU
try:
    import torch
    cuda_available = torch.cuda.is_available()
except:
    cuda_available = False

XGB_PARAMS = dict(
    max_depth=6, learning_rate=0.05, n_estimators=300,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    objective="multi:softprob", num_class=3, random_state=42,
    eval_metric="mlogloss", early_stopping_rounds=25,
    tree_method="hist" if cuda_available else "auto",
)

# ================== Data Fetching ==================
def fetch_candles_chunk(symbol: str, resolution: str, start: int, end: int, retries: int = 3) -> List[Dict]:
    """Fetch one chunk of candles with retry logic."""
    path, url = "/v2/history/candles", BASE_URL + "/v2/history/candles"
    params = {"resolution": RESOLUTION_MAP[resolution], "symbol": symbol, "start": start, "end": end}
    headers = {"Accept": "application/json"}
    
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=20)
            resp.raise_for_status()
            data = resp.json()
            if data.get("success") is False:
                raise ValueError(f"API error: {data}")
            result = data.get("result")
            if isinstance(result, dict):
                return result.get("candles", [])
            return result if isinstance(result, list) else []
        except Exception as exc:
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)
    return []

def fetch_historical_data(symbol: str, resolution: str, limit: int = 50000, batch_size: int = 500) -> pd.DataFrame:
    """Fetch limit candles with batching."""
    if resolution not in INTERVAL_SECONDS:
        raise ValueError(f"Unsupported resolution: {resolution}")
    
    step = INTERVAL_SECONDS[resolution]
    now = int(datetime.now(timezone.utc).timestamp())
    all_candles, end_ts = [], now
    
    log.info("📥 Fetching %s×%s candles for %s", limit, resolution, symbol)
    
    while len(all_candles) < limit:
        fetch_n = min(batch_size, limit - len(all_candles))
        start_ts = end_ts - fetch_n * step
        try:
            chunk = fetch_candles_chunk(symbol, resolution, start_ts, end_ts, retries=3)
        except Exception as exc:
            if all_candles:
                log.warning("Fetch interrupted: %s", exc)
                break
            raise
        
        if not chunk:
            break
        all_candles = chunk + all_candles
        end_ts = start_ts - step
        time.sleep(0.2)
    
    if not all_candles:
        raise RuntimeError(f"Zero candles returned for {symbol} {resolution}")
    
    df = pd.DataFrame(all_candles).rename(columns={"time": "timestamp"})
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df.sort_values("timestamp", inplace=True)
    df.drop_duplicates("timestamp", inplace=True, keep="first")
    df.reset_index(drop=True, inplace=True)
    
    log.info("✓ Fetched %s candles", len(df))
    return df

# ================== Feature Engineering ==================
def add_features(df: pd.DataFrame, resolution: str) -> pd.DataFrame:
    """Compute all 50 canonical features."""
    c, h, l, o, v = df["close"].astype(float), df["high"].astype(float), df["low"].astype(float), df["open"].astype(float), df["volume"].astype(float)
    
    # Price
    for period in [10, 20, 50, 100, 200]:
        df[f"sma_{period}"] = ta.trend.SMAIndicator(c, window=period).sma_indicator()
    df["ema_12"] = ta.trend.EMAIndicator(c, window=12).ema_indicator()
    df["ema_26"] = ta.trend.EMAIndicator(c, window=26).ema_indicator()
    df["ema_50"] = ta.trend.EMAIndicator(c, window=50).ema_indicator()
    
    sma20, sma50, sma200 = df["sma_20"].replace(0, np.nan), df["sma_50"].replace(0, np.nan), df["sma_200"].replace(0, np.nan)
    df["close_sma_20_ratio"], df["close_sma_50_ratio"], df["close_sma_200_ratio"] = c / sma20, c / sma50, c / sma200
    df["high_low_spread"] = (h - l) / l.replace(0, np.nan)
    df["close_open_ratio"] = c / o.replace(0, np.nan)
    df["body_size"] = (c - o).abs() / o.replace(0, np.nan)
    df["upper_shadow"] = (h - np.maximum(o, c)) / h.replace(0, np.nan)
    df["lower_shadow"] = (np.minimum(o, c) - l) / l.replace(0, np.nan)
    
    # Momentum
    df["rsi_14"] = ta.momentum.RSIIndicator(c, window=14).rsi()
    df["rsi_7"] = ta.momentum.RSIIndicator(c, window=7).rsi()
    stoch = ta.momentum.StochasticOscillator(h, l, c, window=14, smooth_window=3)
    df["stochastic_k_14"], df["stochastic_d_14"] = stoch.stoch(), stoch.stoch_signal()
    df["williams_r_14"] = ta.momentum.WilliamsRIndicator(h, l, c, lbp=14).williams_r()
    df["cci_20"] = ta.trend.CCIIndicator(h, l, c, window=20).cci()
    df["roc_10"], df["roc_20"] = ta.momentum.ROCIndicator(c, window=10).roc(), ta.momentum.ROCIndicator(c, window=20).roc()
    df["momentum_10"], df["momentum_20"] = c - c.shift(10), c - c.shift(20)
    
    # Trend
    macd = ta.trend.MACD(c, window_slow=26, window_fast=12, window_sign=9)
    df["macd"], df["macd_signal"], df["macd_histogram"] = macd.macd(), macd.macd_signal(), macd.macd_diff()
    df["adx_14"] = ta.trend.ADXIndicator(h, l, c, window=14).adx()
    aroon = ta.trend.AroonIndicator(high=h, low=l, window=25)
    df["aroon_up"], df["aroon_down"], df["aroon_oscillator"] = aroon.aroon_up(), aroon.aroon_down(), aroon.aroon_indicator()
    df["trend_strength"] = (df["aroon_up"] - df["aroon_down"]).abs()
    
    # Volatility
    bb = ta.volatility.BollingerBands(c, window=20, window_dev=2)
    df["bb_upper"], df["bb_lower"], df["bb_width"] = bb.bollinger_hband(), bb.bollinger_lband(), bb.bollinger_wband()
    df["bb_position"] = (c - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"]).replace(0, np.nan)
    df["atr_14"] = ta.volatility.AverageTrueRange(h, l, c, window=14).average_true_range()
    df["atr_20"] = ta.volatility.AverageTrueRange(h, l, c, window=20).average_true_range()
    df["volatility_10"], df["volatility_20"] = c.pct_change().rolling(10).std().fillna(0), c.pct_change().rolling(20).std().fillna(0)
    
    # Volume
    df["volume_sma_20"] = v.rolling(20).mean()
    df["volume_ratio"] = v / df["volume_sma_20"].replace(0, np.nan)
    df["obv"] = ta.volume.OnBalanceVolumeIndicator(c, v).on_balance_volume()
    df["volume_price_trend"] = (v * c.pct_change()).cumsum()
    mfm = ((c - l) - (h - c)) / (h - l).replace(0, np.nan)
    ad = (mfm * v).cumsum()
    df["accumulation_distribution"] = ad
    df["chaikin_oscillator"] = ad.ewm(span=3, adjust=False).mean() - ad.ewm(span=10, adjust=False).mean()
    
    # Returns
    p1, p24 = RETURNS_1H_BARS.get(resolution, 4), RETURNS_24H_BARS.get(resolution, 96)
    df["returns_1h"], df["returns_24h"] = c.pct_change(p1).fillna(0) * 100, c.pct_change(p24).fillna(0) * 100
    
    return df

def create_labels(df: pd.DataFrame, forward_periods: int = 1) -> np.ndarray:
    """Create labels: 0=SELL, 1=HOLD, 2=BUY."""
    future_close = df["close"].shift(-forward_periods)
    return_pct = (future_close - df["close"]) / df["close"].replace(0, np.nan) * 100
    labels = np.ones(len(df), dtype=int)
    labels[return_pct > BUY_THRESHOLD], labels[return_pct < SELL_THRESHOLD] = 2, 0
    labels[-forward_periods:] = 1
    return labels

def analyze_label_distribution(y: np.ndarray, timeframe: str) -> Dict:
    """Analyze class distribution."""
    counter = Counter(y)
    dist = {"SELL": counter[0], "HOLD": counter[1], "BUY": counter[2]}
    pct = {k: round(v / len(y) * 100, 2) for k, v in dist.items()}
    log.info("[%s] Labels: SELL=%.1f%% HOLD=%.1f%% BUY=%.1f%%", timeframe, pct["SELL"], pct["HOLD"], pct["BUY"])
    return {"dist": dist, "pct": pct}

# ================== Training & Saving ==================
def train_model(X: pd.DataFrame, y: np.ndarray, timeframe: str) -> Tuple[XGBClassifier, Dict]:
    """Train XGBoost with temporal split."""
    n = len(X)
    t1, t2 = int(n * 0.70), int(n * 0.85)
    X_tr, y_tr, X_vl, y_vl, X_te, y_te = X.iloc[:t1], y[:t1], X.iloc[t1:t2], y[t1:t2], X.iloc[t2:], y[t2:]
    
    if len(X_vl) == 0 or len(X_te) == 0:
        raise ValueError(f"Insufficient samples for split")
    
    log.info("[%s] Split: train=%s val=%s test=%s", timeframe, len(X_tr), len(X_vl), len(X_te))
    
    sw_tr = compute_sample_weight("balanced", y_tr)
    fit_params = {k: v for k, v in XGB_PARAMS.items() if k != "early_stopping_rounds"}
    model = XGBClassifier(**fit_params, early_stopping_rounds=XGB_PARAMS["early_stopping_rounds"])
    
    t0 = time.time()
    model.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_vl, y_vl)], verbose=False)
    elapsed = time.time() - t0
    
    metrics = {
        "train_accuracy": float(model.score(X_tr, y_tr)),
        "val_accuracy": float(model.score(X_vl, y_vl)),
        "test_accuracy": float(model.score(X_te, y_te)),
        "training_time": round(elapsed, 2),
        "n_estimators_used": model.best_iteration + 1 if hasattr(model, "best_iteration") else XGB_PARAMS["n_estimators"],
    }
    
    log.info("[%s] Accuracies: train=%.4f val=%.4f test=%.4f", timeframe, metrics["train_accuracy"], metrics["val_accuracy"], metrics["test_accuracy"])
    print(f"\n{'─'*64}\n  {timeframe.upper()} Test Report\n{'─'*64}\n{classification_report(y_te, model.predict(X_te), target_names=['SELL', 'HOLD', 'BUY'], zero_division=0)}\n{'─'*64}")
    
    return model, metrics

def save_model_bundle(model: XGBClassifier, symbol: str, timeframe: str, feature_cols: List[str], metrics: Dict, save_dir: Path) -> Path:
    """Save model with feature validation."""
    if len(feature_cols) != EXPECTED_FEATURE_COUNT:
        raise ValueError(f"Feature count {len(feature_cols)} != expected {EXPECTED_FEATURE_COUNT}")
    if feature_cols != FEATURE_LIST:
        raise ValueError("Features do not match canonical FEATURE_LIST")
    
    save_dir.mkdir(parents=True, exist_ok=True)
    fname, out_path = f"xgboost_classifier_{symbol}_{timeframe}.pkl", save_dir / f"xgboost_classifier_{symbol}_{timeframe}.pkl"
    
    if out_path.exists():
        out_path.rename(out_path.with_suffix(".pkl.bak"))
    
    bundle = {
        "model": model, "feature_cols": feature_cols, "feature_count": len(feature_cols),
        "symbol": symbol, "timeframe": timeframe, "metrics": metrics,
        "trained_at": datetime.now(timezone.utc).isoformat(),
    }
    
    with open(out_path, "wb") as fh:
        pickle.dump(bundle, fh, protocol=pickle.HIGHEST_PROTOCOL)
    
    # Verify
    with open(out_path, "rb") as fh:
        loaded = pickle.load(fh)
    assert isinstance(loaded.get("model"), XGBClassifier)
    dummy = np.random.rand(1, len(feature_cols))
    loaded["model"].predict(dummy)
    
    log.info("✓ Saved & verified → %s", out_path)
    return out_path

# ================== Main Training Loop ==================
print("=" * 80)
print("XGBoost Training (Embedded) — Inline Training Loop")
print("=" * 80)

symbol = os.environ.get("TRAIN_SYMBOL", "BTCUSD")
candles_limit = int(os.environ.get("TRAIN_CANDLES", "50000"))
batch_size = int(os.environ.get("TRAIN_BATCH_SIZE", "500"))
save_dir = Path(os.environ.get("TRAIN_SAVE_DIR", "agent/model_storage/xgboost"))

results = []
for tf in ["15m", "1h", "4h"]:
    print(f"\n{'─'*80}\n  Timeframe: {tf}\n{'─'*80}")
    try:
        df = fetch_historical_data(symbol, tf, limit=candles_limit, batch_size=batch_size)
        if len(df) < 1000:
            log.error("✗ Only %s candles — need ≥1000", len(df))
            continue
        
        df = add_features(df, tf)
        missing = [f for f in FEATURE_LIST if f not in df.columns]
        if missing:
            log.error("✗ Missing features: %s", missing)
            continue
        
        df["label"] = create_labels(df)
        clean = df[FEATURE_LIST + ["label"]].dropna()
        
        if len(clean) < 500:
            log.error("✗ Only %s clean rows", len(clean))
            continue
        
        X, y = clean[FEATURE_LIST], clean["label"].values
        analyze_label_distribution(y, tf)
        
        model, metrics = train_model(X, y, tf)
        path = save_model_bundle(model, symbol, tf, FEATURE_LIST, metrics, save_dir)
        
        results.append({
            "timeframe": tf, "samples": len(X), "features": len(FEATURE_LIST),
            **metrics, "path": str(path)
        })
        print(f"✓ Model saved → {path}")
    except Exception as exc:
        log.error("✗ Failed for %s: %s", tf, exc)

print("\n" + "=" * 80)
if results:
    summary_path = save_dir / "training_summary.csv"
    pd.DataFrame(results).to_csv(summary_path, index=False)
    log.info("✓ Training complete. Summary → %s", summary_path)
else:
    log.error("✗ No models trained")

XGBoost Training (Embedded) — Inline Training Loop

────────────────────────────────────────────────────────────────────────────────
  Timeframe: 15m
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────
  15M Test Report
────────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

        SELL       0.17      0.07      0.10       279
        HOLD       0.96      0.88      0.92      6925
         BUY       0.11      0.42      0.18       255

    accuracy                           0.83      7459
   macro avg       0.41      0.46      0.40      7459
weighted avg       0.90      0.83      0.86      7459

────────────────────────────────────────────────────────────────
✓ Model saved → D:\ATTRAL\Projects\Trading Agent 2/agent/model_storage/xgboost/xgboost_classifier_BTCUSD_15m.pkl

────────────────────────────────────────────────────────────

## 5. Inspect Generated Models

Load and verify the trained models to ensure they were saved correctly and can make predictions.

In [14]:
import pickle
import glob
import os
import numpy as np
from pathlib import Path

# Use XGBOOST_DIR from setup cell
model_dir = XGBOOST_DIR

# List all saved models
model_files = sorted(glob.glob(str(model_dir / "*.pkl")))

if not model_files:
    print("✗ No model files found. Check if training completed successfully.")
    print(f"  Looked in: {model_dir}")
else:
    print(f"✓ Found {len(model_files)} model files:\n")
    
    for model_file in model_files:
        file_name = Path(model_file).name
        file_size_kb = os.path.getsize(model_file) / 1024
        
        try:
            # Load and inspect the model
            with open(model_file, 'rb') as f:
                bundle = pickle.load(f)
            
            model = bundle['model']
            symbol = bundle['symbol']
            timeframe = bundle['timeframe']
            metrics = bundle['metrics']
            n_features = len(bundle['feature_cols'])
            
            print(f"File: {file_name} ({file_size_kb:.1f} KB)")
            print(f"  Symbol: {symbol}, Timeframe: {timeframe}")
            print(f"  Features: {n_features}")
            print(f"  Accuracy: train={metrics['train_accuracy']:.4f}, "
                  f"val={metrics['val_accuracy']:.4f}, test={metrics['test_accuracy']:.4f}")
            print(f"  Training Time: {metrics['training_time']}s")
            
            # Test prediction
            dummy_data = np.random.rand(5, n_features)
            predictions = model.predict(dummy_data)
            probabilities = model.predict_proba(dummy_data)
            print(f"  Test Prediction: ✓ Works (sample predictions: {predictions})")
            print()
            
        except Exception as e:
            print(f"✗ Error loading {file_name}: {e}\n")

✓ Found 3 model files:

File: xgboost_classifier_BTCUSD_15m.pkl (2878.7 KB)
  Symbol: BTCUSD, Timeframe: 15m
  Features: 50
  Accuracy: train=0.9158, val=0.8475, test=0.8342
  Training Time: 17.24s
  Test Prediction: ✓ Works (sample predictions: [1 1 1 1 1])

File: xgboost_classifier_BTCUSD_1h.pkl (2782.0 KB)
  Symbol: BTCUSD, Timeframe: 1h
  Features: 50
  Accuracy: train=0.8877, val=0.8490, test=0.6588
  Training Time: 10.6s
  Test Prediction: ✓ Works (sample predictions: [2 2 2 2 1])

File: xgboost_classifier_BTCUSD_4h.pkl (1244.7 KB)
  Symbol: BTCUSD, Timeframe: 4h
  Features: 50
  Accuracy: train=0.9199, val=0.5762, test=0.4158
  Training Time: 3.68s
  Test Prediction: ✓ Works (sample predictions: [2 2 2 2 2])



## 6. Review Training Summary

Display the training summary CSV file with key metrics for all trained models.

In [ ]:
import pandas as pd
import os
from pathlib import Path

# Use TRAINING_SUMMARY_FILE from setup cell
summary_file = TRAINING_SUMMARY_FILE

if summary_file.exists():
    df = pd.read_csv(summary_file)
    print("Training Summary Report:")
    print("=" * 100)
    print(df.to_string(index=False))
    print("=" * 100)
    
    # Summary statistics
    print("\nSummary Statistics:")
    print(f"  Total Models Trained: {len(df)}")
    print(f"  Average Test Accuracy: {df['test_accuracy'].mean():.4f}")
    print(f"  Best Test Accuracy: {df['test_accuracy'].max():.4f} ({df.loc[df['test_accuracy'].idxmax(), 'timeframe']})")
    print(f"  Total Training Time: {df['training_time'].sum():.1f}s")
else:
    print(f"✗ Summary file not found: {summary_file}")
    print(f"  Run the training cell first to generate this file.")

✗ Summary file not found: D:\ATTRAL\Projects\Trading Agent 2/agent/model_storage/xgboost/training_summary.csv
  Run the training cell first to generate this file.


## Troubleshooting

### Issue: ModuleNotFoundError or missing packages
Solution: Re-run the Install Dependencies cell above.

### Issue: Only X candles - need >= 1000
Solution: The API may be rate-limiting or the symbol lacks historical data.
- Try reducing TRAIN_CANDLES to 10000 or 5000
- Verify the symbol is correct (BTCUSD for Bitcoin)

### Issue: Training is very slow or runs out of memory
Solution: Reduce the candle count or enable GPU:
- Set TRAIN_CANDLES to a smaller value (10000 or 15000)
- Ensure GPU is enabled (check Environment Variables cell)
- In Colab: Runtime -> Change Runtime Type -> GPU

### Issue: API errors or connection timeout
Solution: Delta Exchange API may be temporarily unavailable. Wait a few minutes and re-run.

### Issue: Models not saving locally
Solution: Verify the save directory has write permissions:
1. Check TRAIN_SAVE_DIR path in Environment Variables
2. Ensure sufficient storage space in Colab
3. Check output messages for specific errors


## 3. Results & Next Steps

After successful training:

1. **Download Models:** Models can be downloaded from Colab Files section in:/
   - agent/model_storage/xgboost/*.pkl files

2. **Use Models in Agent:** Integrate downloaded models with your local agent

3. **Adjust Parameters:** Experiment with different settings:
   - Increase TRAIN_CANDLES for more training data
   - Adjust buy/sell thresholds in the script for different sensitivity
   - Change timeframes in the training script

4. **Monitor Performance:** Check training_summary.csv for model accuracy metrics

5. **Retrain Periodically:** Retrain models monthly with fresh market data to maintain accuracy

---

For more information:
- XGBoost docs: https://xgboost.readthedocs.io/
- TA-Lib indicators: https://github.com/bukosabino/ta
- Delta Exchange API: https://docs.delta.exchange/


## Delta Historical Fetcher Integration

This section integrates the `delta_historical_fetcher.py` utility into the Colab workflow. It attempts to load the script from `"/content/drive/My Drive/Trading Agent 2/scripts/delta_historical_fetcher.py"` and run a small smoke-test fetch. If the file is not present in your Drive, upload it to that path or adjust the `fetcher_path` variable below.

In [ ]:
# Integration test for delta_historical_fetcher
import os
import importlib.util
from pathlib import Path

# Use paths from setup cell
fetcher_path = DELTA_FETCHER_SCRIPT
out_dir = DATA_DELTA_DIR

if not fetcher_path.exists():
    print('✗ Fetcher script not found at:', fetcher_path)
    print('  Please ensure `scripts/delta_historical_fetcher.py` is present in your project.')
else:
    spec = importlib.util.spec_from_file_location('delta_fetcher', str(fetcher_path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    print('✓ Loaded delta_historical_fetcher from:', fetcher_path)

    # Instantiate the fetcher and run a short smoke-fetch (1 day of 1h candles)
    try:
        fetcher = mod.DeltaHistoricalDataFetcher(env='prod')
        df = fetcher.fetch_range_by_dates('BTCUSD', '1h', '2024-01-01', '2024-01-02')
        if df.empty:
            print('✗ No data returned for test fetch. Check API availability / symbol.')
        else:
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / 'btcusd_1h_test.csv'
            fetcher.save_to_csv(df, str(out_path))
            print(f'✓ Test fetch saved → {out_path} (rows: {len(df)})')
            display(df.head())
    except Exception as e:
        print('✗ Error during fetcher test:', e)

✗ fetcher script not found at: /content/drive/My Drive/Trading Agent 2/scripts/delta_historical_fetcher.py
  Please ensure `scripts/delta_historical_fetcher.py` is present in your Drive.


## Automatic Feature-Consistency Check

This cell loads the canonical `FEATURE_LIST` from the training script and verifies that all saved model bundles in `TRAIN_SAVE_DIR` use exactly the same 50 features in the same order. If checks pass, it writes a `feature_schema.json` file in the save directory for downstream agents to consume.

In [ ]:
# Feature-consistency verification
import os
import glob
import pickle
import json
from pathlib import Path
from datetime import datetime, timezone

# Canonical 50-feature list (must match training code above)
CANONICAL_FEATURES = [
    "sma_10", "sma_20", "sma_50", "sma_100", "sma_200",
    "ema_12", "ema_26", "ema_50",
    "close_sma_20_ratio", "close_sma_50_ratio", "close_sma_200_ratio",
    "high_low_spread", "close_open_ratio", "body_size", "upper_shadow", "lower_shadow",
    "rsi_14", "rsi_7", "stochastic_k_14", "stochastic_d_14",
    "williams_r_14", "cci_20", "roc_10", "roc_20",
    "momentum_10", "momentum_20",
    "macd", "macd_signal", "macd_histogram",
    "adx_14", "aroon_up", "aroon_down", "aroon_oscillator",
    "trend_strength",
    "bb_upper", "bb_lower", "bb_width", "bb_position",
    "atr_14", "atr_20",
    "volatility_10", "volatility_20",
    "volume_sma_20", "volume_ratio", "obv",
    "volume_price_trend", "accumulation_distribution", "chaikin_oscillator",
    "returns_1h", "returns_24h",
]
EXPECTED_COUNT = len(CANONICAL_FEATURES)

# Use XGBOOST_DIR from setup cell
save_dir = XGBOOST_DIR

print(f"✓ Canonical features: {EXPECTED_COUNT} features")
print(f"✓ Model directory: {save_dir}")

# Find saved bundles
if not save_dir.exists():
    print('✗ Save directory does not exist:', save_dir)
else:
    bundles = sorted(glob.glob(str(save_dir / '*.pkl')))
    if not bundles:
        print('✗ No model bundles found in', save_dir)
    else:
        mismatches = []
        for b in bundles:
            try:
                with open(b, 'rb') as fh:
                    bundle = pickle.load(fh)
            except Exception as e:
                print(f'✗ Failed to load {Path(b).name}: {e}')
                mismatches.append((b, 'load-failed'))
                continue

            fcols = bundle.get('feature_cols', [])
            if not fcols:
                print(f'✗ {Path(b).name}: missing feature_cols')
                mismatches.append((b, 'missing'))
                continue

            if len(fcols) != EXPECTED_COUNT:
                print(f'✗ {Path(b).name}: feature count {len(fcols)} != {EXPECTED_COUNT}')
                mismatches.append((b, 'count-mismatch'))
            elif fcols != CANONICAL_FEATURES:
                print(f'✗ {Path(b).name}: feature list mismatch (order or names differ)')
                mismatches.append((b, 'list-mismatch'))
            else:
                print(f'✓ {Path(b).name}: features OK ({len(fcols)} features)')

        if not mismatches:
            schema = {
                'features': CANONICAL_FEATURES,
                'feature_count': EXPECTED_COUNT,
                'generated_at': datetime.now(timezone.utc).isoformat()
            }
            schema_path = save_dir / 'feature_schema.json'
            with open(schema_path, 'w') as fh:
                json.dump(schema, fh, indent=2)
            print(f'\n✓ All {len(bundles)} bundles validated. Wrote feature_schema.json')
        else:
            print(f'\n✗ {len(mismatches)} bundle(s) have issues. See above.')

✓ Canonical features: 50 features
✗ Save directory does not exist: /content/drive/My Drive/Trading Agent 2/agent/model_storage/xgboost
